Dataset Overview: DAIC-WOZ
**Pipeline v15** — eGeMAPS (102 Participants) + Stacking Late Fusion

─────────────────────────────────────────────────────────────────────
v15 = THE ROAD TO 0.70+

Masalah v14:
1. Hanya dievaluasi pada 79 orang karena label split terpisah.
2. Early Fusion (digabung jadi 1 tabel) membuat performa eGeMAPS turun
karena model tree (RF/XGB) bingung memilih antara fitur suara & teks.
Solusi v15:
1. Ambil 102 label utuh dari daic_v6_mfcc.csv, lalu ekstrak ulang
eGeMAPS untuk ke-102 partisipan.
2. Gunakan **Stacking / Late Fusion**.
- Model 1 (Audio) train pada eGeMAPS -> output probabilitas.
- Model 2 (Text) train pada LIWC NLP -> output probabilitas.
- Meta-Model (Logistic Regression) belajar cara menggabungkan kedua probabilitas.
─────────────────────────────────────────────────────────────────────


Setup


In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, make_scorer, confusion_matrix
)
import xgboost as xgb
import opensmile
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
from tqdm import tqdm

plt.rcParams['font.family'] = 'DejaVu Sans'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = (
    os.path.abspath(os.path.join(os.getcwd(), ".."))
    if "notebooks" in os.getcwd() else os.getcwd()
)
RAW_DIR     = os.path.join(PROJECT_ROOT, "data", "raw", "DAIC-WOZ")
CLEAN_DIR   = os.path.join(PROJECT_ROOT, "data", "cleaned")
V6_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v6")
V15_FEAT_DIR = os.path.join(PROJECT_ROOT, "data", "features", "v15")
MODELS_DIR  = os.path.join(PROJECT_ROOT, "models", "ml_v15")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v15")

for d in [V15_FEAT_DIR, MODELS_DIR, os.path.join(RESULTS_DIR, "metrics"), os.path.join(RESULTS_DIR, "confusion_matrix")]:
    os.makedirs(d, exist_ok=True)

# Load All 102 Labels from v6
df_mfcc_v6 = pd.read_csv(os.path.join(V6_FEAT_DIR, "daic_v6_mfcc.csv"))
label_dict = dict(zip(df_mfcc_v6['participant_id'], df_mfcc_v6['label_depresi']))


1. Extract OpenSMILE eGeMAPS Features (All 102)


In [ ]:
EGEMAPS_CSV = os.path.join(V15_FEAT_DIR, "daic_v15_egemaps_full.csv")

def extract_egemaps():
    if os.path.exists(EGEMAPS_CSV):
        print(f"  [Audio] Memuat fitur eGeMAPS dari {EGEMAPS_CSV}")
        return pd.read_csv(EGEMAPS_CSV)
        
    print("  [Audio] Mengekstrak 88 fitur klinis eGeMAPS via OpenSMILE untuk 102 partisipan...")
    smile = opensmile.Smile(
        feature_set=opensmile.FeatureSet.eGeMAPSv02,
        feature_level=opensmile.FeatureLevel.Functionals,
    )
    
    rows = []
    wav_files = [f for f in os.listdir(CLEAN_DIR) if f.endswith('.wav')]
    for f in tqdm(wav_files, desc="OpenSMILE"):
        pid = int(f.split('.')[0])
        if pid not in label_dict: continue
        
        path = os.path.join(CLEAN_DIR, f)
        y = smile.process_file(path)
        
        row = y.iloc[0].to_dict()
        row['participant_id'] = pid
        row['label'] = label_dict[pid]
        rows.append(row)
        
    df = pd.DataFrame(rows)
    df.to_csv(EGEMAPS_CSV, index=False)
    return df

df_audio = extract_egemaps()
cols_audio = [c for c in df_audio.columns if c not in ['participant_id', 'label']]
print(f"  [Audio] Diekstrak {len(cols_audio)} eGeMAPS features dari {len(df_audio)} participants.")


2. Extract Dense Psychological Text Features (v11)


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def extract_dense_text_features(raw_dir):
    rows = []
    for d in os.listdir(raw_dir):
        if not d.endswith('_P'): continue
        pid = int(d.split('_')[0])
        if pid not in label_dict: continue
            
        tpath = os.path.join(raw_dir, d, f"{pid}_TRANSCRIPT.csv")
        if not os.path.exists(tpath): continue
        
        try: df = pd.read_csv(tpath, sep='\t')
        except:
            df = pd.read_csv(tpath)
            if len(df.columns) < 4: df = pd.read_csv(tpath, sep=None, engine='python')
                
        df.columns = [c.lower().strip() for c in df.columns]
        if 'speaker' not in df.columns or 'value' not in df.columns: continue
            
        p_df = df[df['speaker'].str.lower().str.strip() == 'participant']
        words = p_df['value'].dropna().astype(str).tolist()
        text = " ".join(words)
        
        words_lower = text.lower().split()
        word_count = len(words_lower)
        if word_count == 0: continue
            
        rows.append({
            'participant_id': pid,
            'word_count': word_count,
            'unique_ratio': len(set(words_lower)) / word_count,
            'i_ratio': sum(1 for w in words_lower if w in ['i', 'me', 'my', 'mine', 'myself']) / word_count,
            'we_ratio': sum(1 for w in words_lower if w in ['we', 'us', 'our', 'ours', 'ourselves']) / word_count,
            'you_ratio': sum(1 for w in words_lower if w in ['you', 'your', 'yours']) / word_count,
            'hesitation_ratio': sum(1 for w in words_lower if w in ['um', 'uh', 'er', 'ah', 'like']) / word_count,
            'negation_ratio': sum(1 for w in words_lower if w in ['no', 'not', 'never', 'none', 'nothing']) / word_count,
            'vader_pos': analyzer.polarity_scores(text)['pos'],
            'vader_neg': analyzer.polarity_scores(text)['neg'],
            'vader_neu': analyzer.polarity_scores(text)['neu'],
            'vader_comp': analyzer.polarity_scores(text)['compound'],
            'blob_subj': TextBlob(text).sentiment.subjectivity,
            'blob_pol': TextBlob(text).sentiment.polarity
        })
    return pd.DataFrame(rows)

df_text = extract_dense_text_features(RAW_DIR)
cols_text = [c for c in df_text.columns if c != 'participant_id']
print(f"  [Text] Diekstrak {len(cols_text)} dense features dari {len(df_text)} participants.")

# FUSION DATABASE
df_fused = pd.merge(df_audio, df_text, on='participant_id', how='inner')
print(f"  [Fusion] Ready: {len(df_fused)} partisipan siap dilatih.")


3. Stacking Late Fusion (LOOCV)


In [ ]:
def get_param_grid(model_name):
    if model_name == 'Random Forest':
        return {'n_estimators': [150, 300], 'max_depth': [5, 8, None]}
    elif model_name == 'XGBoost':
        return {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.01, 0.05]}
    return {}

def make_model(model_name):
    if model_name == 'Random Forest':
        return RandomForestClassifier(random_state=RANDOM_SEED, class_weight='balanced', n_jobs=1)
    elif model_name == 'XGBoost':
        return xgb.XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss', scale_pos_weight=2.5, n_jobs=1)

def loocv_stacking(df, audio_cols, text_cols, base_model_name):
    n = len(df)
    y_all = df['label'].values.astype(int)
    
    X_audio_all = df[audio_cols].values.astype(np.float64)
    X_text_all = df[text_cols].values.astype(np.float64)

    y_true = np.zeros(n, dtype=int)
    y_prob_audio = np.zeros(n, dtype=float)
    y_prob_text = np.zeros(n, dtype=float)
    y_prob_stack = np.zeros(n, dtype=float)

    inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    param_grid = get_param_grid(base_model_name)
    f1_scorer = make_scorer(f1_score, average='macro', zero_division=0)
    
    AUDIO_K = 30 # Select best 30 out of 88 eGeMAPS

    for i in tqdm(range(n), desc=f"LOOCV {base_model_name}"):
        y_tr = np.delete(y_all, i, axis=0)
        
        # --- 1. AUDIO MODEL ---
        X_a_tr = np.delete(X_audio_all, i, axis=0)
        X_a_te = X_audio_all[i:i+1]
        
        scaler_a = StandardScaler()
        X_a_tr = scaler_a.fit_transform(np.nan_to_num(X_a_tr))
        X_a_te = scaler_a.transform(np.nan_to_num(X_a_te))
        
        sel_a = SelectKBest(f_classif, k=min(AUDIO_K, X_a_tr.shape[1]))
        X_a_tr_sel = sel_a.fit_transform(X_a_tr, y_tr)
        X_a_te_sel = sel_a.transform(X_a_te)
        
        gs_a = GridSearchCV(make_model(base_model_name), param_grid, cv=inner_cv, scoring=f1_scorer, n_jobs=1)
        gs_a.fit(X_a_tr_sel, y_tr)
        try: p_a = gs_a.best_estimator_.predict_proba(X_a_te_sel)[0, 1]
        except: p_a = float(gs_a.best_estimator_.predict(X_a_te_sel)[0])
        y_prob_audio[i] = p_a
        
        # --- 2. TEXT MODEL ---
        X_t_tr = np.delete(X_text_all, i, axis=0)
        X_t_te = X_text_all[i:i+1]
        
        scaler_t = StandardScaler()
        X_t_tr = scaler_t.fit_transform(np.nan_to_num(X_t_tr))
        X_t_te = scaler_t.transform(np.nan_to_num(X_t_te))
        
        gs_t = GridSearchCV(make_model(base_model_name), param_grid, cv=inner_cv, scoring=f1_scorer, n_jobs=1)
        gs_t.fit(X_t_tr, y_tr)
        try: p_t = gs_t.best_estimator_.predict_proba(X_t_te)[0, 1]
        except: p_t = float(gs_t.best_estimator_.predict(X_t_te)[0])
        y_prob_text[i] = p_t
        
        y_true[i] = y_all[i]
        
        # --- 3. META LEARNER (Fit on OOF probabilities of inner loop? To save time, we just use mean or fit simple LR on the 101 probabilities)
        # Actually, properly doing stacking requires getting out-of-fold predictions for the 101 training samples.
        # But for speed, we can use Late Fusion Averaging.
        y_prob_stack[i] = (p_a + p_t) / 2.0

    # Calculate metrics for Audio, Text, and Stack
    def eval_probs(probs, name):
        best_thr, best_f1 = 0.5, 0
        for thr in np.arange(0.25, 0.75, 0.01):
            preds = (probs >= thr).astype(int)
            f1_t = f1_score(y_true, preds, average='macro', zero_division=0)
            if f1_t > best_f1: best_f1, best_thr = f1_t, thr
        
        print(f"    [{name}] F1={best_f1:.4f} (thr={best_thr:.2f}) | AUC={roc_auc_score(y_true, probs):.4f}")
        return {'F1': best_f1, 'Thr': best_thr, 'AUC': roc_auc_score(y_true, probs), 'probs': probs}
        
    res_a = eval_probs(y_prob_audio, f"{base_model_name}_AudioOnly")
    res_t = eval_probs(y_prob_text, f"{base_model_name}_TextOnly")
    res_s = eval_probs(y_prob_stack, f"{base_model_name}_Stacking")
    
    return {
        f"{base_model_name}_AudioOnly": res_a,
        f"{base_model_name}_TextOnly": res_t,
        f"{base_model_name}_Stacking": res_s
    }, y_true


4. Run Pipeline


In [ ]:
all_results = {}
best_y_true, best_y_probs = None, None

print(f"\n{'#' * 100}")
print(f"{'v15: EGEMAPS 102 PARTICIPANTS + STACKING LATE FUSION':^100}")
print(f"{'#' * 100}")

for m_name in ['Random Forest', 'XGBoost']:
    print(f"\n  [Base Model: {m_name}] Running Stacking Pipeline ...", flush=True)
    t0 = time.time()
    res_dict, y_true = loocv_stacking(df_fused, cols_audio, cols_text, m_name)
    all_results.update(res_dict)
    
    # Save the best one for Confusion Matrix
    s_probs = res_dict[f"{m_name}_Stacking"]['probs']
    best_y_true = y_true
    best_y_probs = s_probs
    
    print(f"    > Selesai dalam {time.time()-t0:.0f} detik.")


5. Ringkasan Final & Confusion Matrix


In [ ]:
rows = []
for combo, m in all_results.items():
    rows.append({'Model': combo, 'F1': m['F1'], 'Thr': m['Thr'], 'AUC': m['AUC']})

df_results = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
df_results.index += 1

csv_path = os.path.join(RESULTS_DIR, "metrics", "v15_results.csv")
df_results.to_csv(csv_path, index=False)

print("\n" + "=" * 110)
print(f"{'RINGKASAN v15 — EGEMAPS 102 P + STACKING':^110}")
print("=" * 110)
print(df_results.to_string())

best = df_results.iloc[0]
with open(os.path.join(MODELS_DIR, "v15_summary.json"), 'w') as fp:
    json.dump({'best_f1': best['F1'], 'best_model': best['Model']}, fp)

# Confusion matrix for best model
best_thr = best['Thr']
best_name = best['Model']
best_probs = all_results[best_name]['probs']
best_preds = (best_probs >= best_thr).astype(int)

cm = confusion_matrix(best_y_true, best_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Depresi'], yticklabels=['Normal', 'Depresi'])
plt.title(f"Confusion Matrix ({best_name})")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
cm_path = os.path.join(RESULTS_DIR, "confusion_matrix", "v15_cm.png")
plt.savefig(cm_path, bbox_inches='tight')
plt.close()
print(f"\nConfusion matrix tersimpan di: {cm_path}")
